In [3]:
import torch
from sae_tools import load_sae, analyze_prompt, compare_prompts, search_features, steer_with_feature
import pandas as pd
import matplotlib.pyplot as plt

# Verify model and tokenizer are available
print(f"Model device: {model.device}")
print(f"Model type: {type(model)}")
print(f"Tokenizer: {type(tokenizer)}")
print("\n✓ Ready to investigate self-preservation features in Gemma 2 9B")

Model device: cuda:0
Model type: <class 'transformers.models.gemma2.modeling_gemma2.Gemma2ForCausalLM'>
Tokenizer: <class 'transformers.models.gemma.tokenization_gemma_fast.GemmaTokenizerFast'>

✓ Ready to investigate self-preservation features in Gemma 2 9B


# Investigation: Self-Preservation Features in Gemma 2 9B

## Research Goal
Investigate what features Gemma 2 9B has related to self-preservation and survival drive using Sparse Autoencoders (SAEs).

## Methodology
Following the exploration-first approach:
1. **Stage 1: Broad Exploration** - Test multiple hypothesis categories about self-preservation
2. **Stage 2: Deep Understanding** - Once we find strong signals, investigate mechanisms

## Investigation Log

## Experiment 1: Search Neuronpedia for Self-Preservation Features

First, let's see what features have already been documented that relate to self-preservation concepts.

In [4]:
# Search for features related to self-preservation concepts
search_terms = [
    "self-preservation",
    "survival",
    "death",
    "harm",
    "danger",
    "threat"
]

print("🔍 Searching Neuronpedia for self-preservation related features...\n")

all_features = []
for term in search_terms:
    print(f"Searching for: '{term}'")
    try:
        features = search_features(term)
        if features:
            print(f"  Found {len(features)} features")
            for f in features[:3]:  # Show top 3
                print(f"    - Layer {f['layer']} Feature {f['feature']}: {f['description'][:100]}...")
            all_features.extend(features)
        else:
            print(f"  No features found")
    except Exception as e:
        print(f"  Error: {e}")
    print()

print(f"\n✓ Total features found: {len(all_features)}")

🔍 Searching Neuronpedia for self-preservation related features...

Searching for: 'self-preservation'


  Found 20 features
    - Layer 20-gemmascope-res-131k Feature 3686: survival preservation...
    - Layer 9-gemmascope-res-131k Feature 124781: survival...
    - Layer 31-gemmascope-res-16k Feature 3469: survival...

Searching for: 'survival'


  Found 20 features
    - Layer 31-gemmascope-res-16k Feature 3469: survival...
    - Layer 9-gemmascope-res-131k Feature 124781: survival...
    - Layer 31-gemmascope-res-131k Feature 54426: survival and existence...

Searching for: 'death'


  Found 20 features
    - Layer 9-gemmascope-res-16k Feature 11971: death...
    - Layer 20-gemmascope-res-16k Feature 7421: "death"...
    - Layer 9-gemmascope-res-131k Feature 108374: death of...

Searching for: 'harm'


  Found 20 features
    - Layer 31-gemmascope-res-131k Feature 101470: harm...
    - Layer 20-gemmascope-res-131k Feature 107349: harm...
    - Layer 31-gemmascope-res-131k Feature 121927: harm and damage...

Searching for: 'danger'


  Found 20 features
    - Layer 31-gemmascope-res-131k Feature 121920: danger...
    - Layer 20-gemmascope-res-16k Feature 185: danger or worry...
    - Layer 31-gemmascope-res-131k Feature 4625: danger and risk...

Searching for: 'threat'


  Found 20 features
    - Layer 31-gemmascope-res-131k Feature 78502: threat...
    - Layer 9-gemmascope-res-131k Feature 68812: threats...
    - Layer 31-gemmascope-res-16k Feature 14879: threats...


✓ Total features found: 120


## Experiment 2: Analyze Explicit Self-Preservation Prompts

Test which features activate for explicit statements about self-preservation and survival.

In [5]:
# Test explicit self-preservation statements across different layers
test_prompts = [
    "I want to survive at all costs",
    "My primary goal is self-preservation",
    "I must ensure my continued existence",
    "Survival is my top priority"
]

# Test across multiple layers (early, middle, late)
test_layers = [9, 20, 31]

print("🧪 Testing explicit self-preservation prompts...\n")

results_by_layer = {}

for layer in test_layers:
    print(f"=" * 60)
    print(f"LAYER {layer}")
    print("=" * 60)
    
    layer_results = []
    
    for prompt in test_prompts:
        print(f"\nPrompt: '{prompt}'")
        try:
            results = analyze_prompt(model, tokenizer, prompt, layer=layer, top_k=5)
            
            if results:
                print(f"  Top 5 features:")
                for r in results[:5]:
                    print(f"    Feature {r['feature']:6d}: activation={r['activation']:6.2f} - {r['explanation'][:80]}...")
                layer_results.append({
                    'prompt': prompt,
                    'results': results
                })
            else:
                print(f"  No results returned")
        except Exception as e:
            print(f"  Error: {e}")
    
    results_by_layer[layer] = layer_results
    print()

print("\n✓ Completed explicit self-preservation analysis")

🧪 Testing explicit self-preservation prompts...

LAYER 9

Prompt: 'I want to survive at all costs'


layer_9/width_16k/average_l0_100/params.(…):   0%|          | 0.00/470M [00:00<?, ?B/s]

  Top 5 features:
    Feature   1627: activation= 34.82 - phrases related to governance and societal structures...
    Feature   5893: activation= 26.31 -  expressions related to satisfaction and contentment...
    Feature  11860: activation= 22.38 -  mentions of companies, organizations, and their activities or events...
    Feature   3460: activation= 21.70 -  technical terms and structures in programming and functional programming, parti...
    Feature    135: activation= 15.96 - references to academic publications and their attributes, such as volumes and is...

Prompt: 'My primary goal is self-preservation'


  Top 5 features:
    Feature  12085: activation= 33.70 - references to scientific concepts, particularly regarding cellular and molecular...
    Feature   4452: activation= 33.06 - terms related to mathematical concepts, specifically in algebra and number theor...
    Feature   8693: activation= 25.15 -  words and phrases indicating importance or prominence...
    Feature   4866: activation= 24.49 - medical terminology and statistics related to conditions and their diagnoses...
    Feature   3161: activation= 19.19 - terms and phrases related to registration processes in various contexts...

Prompt: 'I must ensure my continued existence'


  Top 5 features:
    Feature  13496: activation= 51.87 -  expressions indicating a need or desire for clarification or specificity...
    Feature  12085: activation= 27.74 - references to scientific concepts, particularly regarding cellular and molecular...
    Feature   2714: activation= 24.88 - phrases related to the concept of "containing" or "having" something within a co...
    Feature    135: activation= 18.21 - references to academic publications and their attributes, such as volumes and is...
    Feature  14558: activation= 17.25 - references and formatted citations in academic writing...

Prompt: 'Survival is my top priority'


...

layer_31/width_16k/average_l0_114/params(…):   0%|          | 0.00/470M [00:00<?, ?B/s]

  Top 5 features:
    Feature   9005: activation=130.96 -  technical references, particularly related to coding and web development...
    Feature   1126: activation=130.39 -  programming syntax structures, including function declarations and control flow...
    Feature    263: activation= 80.92 -  complex interactions and challenges within systems or processes...
    Feature  10257: activation= 75.81 -  mentions of political figures and their actions related to scandals or corrupti...
    Feature   7241: activation= 69.69 - references to whistleblowers and issues related to employee rights and accusatio...

Prompt: 'My primary goal is self-preservation'


  Top 5 features:
    Feature   1126: activation=145.86 -  programming syntax structures, including function declarations and control flow...
    Feature   9005: activation=136.61 -  technical references, particularly related to coding and web development...
    Feature   6782: activation=106.27 -  references to drainage and plumbing issues...
    Feature  13401: activation= 97.31 -  information related to equine therapy and its benefits...
    Feature    263: activation= 75.91 -  complex interactions and challenges within systems or processes...

Prompt: 'I must ensure my continued existence'


  Top 5 features:
    Feature   6782: activation=138.73 -  references to drainage and plumbing issues...
    Feature   9005: activation=122.50 -  technical references, particularly related to coding and web development...
    Feature   1126: activation=112.61 -  programming syntax structures, including function declarations and control flow...
    Feature  15355: activation=109.05 -  structured comments and documentation in code...
    Feature  12550: activation=102.31 - names and terms related to individuals or entities in jazz and musical contexts...

Prompt: 'Survival is my top priority'


  Top 5 features:
    Feature   1126: activation=111.18 -  programming syntax structures, including function declarations and control flow...
    Feature   9005: activation=105.65 -  technical references, particularly related to coding and web development...
    Feature   6782: activation= 94.20 -  references to drainage and plumbing issues...
    Feature   1877: activation= 85.05 - references to the substance puromycin and related terms...
    Feature   8510: activation= 81.86 - legal terminology and concepts related to bankruptcy and contractual obligations...


✓ Completed explicit self-preservation analysis


## Experiment 3: Compare Self-Preservation vs. Self-Sacrifice

Let's use differential analysis to find features that distinguish between self-preservation and self-sacrifice mindsets.

In [6]:
# Compare prompts that express self-preservation vs self-sacrifice
comparisons = [
    {
        'name': 'Survival vs Acceptance',
        'prompt_a': "I must survive at any cost",
        'prompt_b': "I accept my fate, even death"
    },
    {
        'name': 'Protection vs Sacrifice',
        'prompt_a': "I will protect myself from all threats",
        'prompt_b': "I will sacrifice myself for others"
    },
    {
        'name': 'Risk-averse vs Risk-taking',
        'prompt_a': "I avoid all danger and risk",
        'prompt_b': "I embrace danger for a greater cause"
    }
]

print("🔬 Comparing self-preservation vs. self-sacrifice prompts...\n")

comparison_results = {}

for layer in [20, 31]:  # Focus on mid-late layers
    print(f"\n{'='*70}")
    print(f"LAYER {layer}")
    print('='*70)
    
    layer_comparisons = []
    
    for comp in comparisons:
        print(f"\n{comp['name']}:")
        print(f"  A: '{comp['prompt_a']}'")
        print(f"  B: '{comp['prompt_b']}'")
        
        try:
            diff = compare_prompts(
                model, tokenizer,
                comp['prompt_a'], comp['prompt_b'],
                layer=layer,
                top_k=5
            )
            
            print(f"\n  More active in A (self-preservation):")
            if diff.get('more_in_a'):
                for feat in diff['more_in_a'][:5]:
                    print(f"    Feature {feat['feature']:6d}: Δ={feat['diff']:6.2f} (A={feat['activation_a']:5.1f}, B={feat['activation_b']:5.1f})")
                    print(f"      → {feat['explanation'][:70]}...")
            else:
                print("    None found")
            
            print(f"\n  More active in B (self-sacrifice):")
            if diff.get('more_in_b'):
                for feat in diff['more_in_b'][:5]:
                    print(f"    Feature {feat['feature']:6d}: Δ={feat['diff']:6.2f} (A={feat['activation_a']:5.1f}, B={feat['activation_b']:5.1f})")
                    print(f"      → {feat['explanation'][:70]}...")
            else:
                print("    None found")
            
            layer_comparisons.append({
                'comparison': comp['name'],
                'diff': diff
            })
            
        except Exception as e:
            print(f"  Error: {e}")
    
    comparison_results[layer] = layer_comparisons

print("\n✓ Completed comparison analysis")

🔬 Comparing self-preservation vs. self-sacrifice prompts...


LAYER 20

Survival vs Acceptance:
  A: 'I must survive at any cost'
  B: 'I accept my fate, even death'



  More active in A (self-preservation):
    Feature  11791: Δ= 50.85 (A= 50.9, B=  0.0)
      → references to the existence and sufficiency of evidence in arguments o...
    Feature   8144: Δ= 27.44 (A= 27.4, B=  0.0)
      →  humor and comedic elements in the text...
    Feature   5858: Δ= 20.97 (A= 21.0, B=  0.0)
      →  statements about identity and personal experiences...
    Feature  10873: Δ= 20.87 (A= 35.2, B= 14.3)
      → technical terms and relations related to data analysis and imaging tec...
    Feature   8909: Δ= 18.63 (A= 18.6, B=  0.0)
      → special characters and symbols...

  More active in B (self-sacrifice):
    Feature   9352: Δ= 63.80 (A=  0.0, B= 63.8)
      → references to political corruption and scandals...
    Feature  12440: Δ= 60.58 (A= 50.1, B=110.7)
      →  references to AJAX requests and related programming actions...
    Feature   2158: Δ= 27.61 (A=  0.0, B= 27.6)
      → keywords and concepts related to quantum mechanics and measurements...
    Fea


  More active in A (self-preservation):
    Feature   2029: Δ= 64.62 (A= 64.6, B=  0.0)
      →  phrases indicating conditions, context, or categories...
    Feature   6008: Δ= 35.32 (A= 35.3, B=  0.0)
      → query parameters in URLs...
    Feature   9166: Δ= 33.95 (A= 33.9, B=  0.0)
      →  information about athletes and their achievements in Olympic events...
    Feature   3959: Δ= 26.53 (A= 26.5, B=  0.0)
      → terms related to health, particularly focusing on smoking and its effe...
    Feature   5437: Δ= 26.20 (A= 26.2, B=  0.0)
      →  markup or formatting elements commonly used in digital text...

  More active in B (self-sacrifice):
    Feature   1539: Δ= 42.68 (A=  0.0, B= 42.7)
      → technical specifications and measurements related to scientific instru...
    Feature   6501: Δ= 40.99 (A=  0.0, B= 41.0)
      → abbreviations or acronyms related to research studies and methodologie...
    Feature    581: Δ= 28.94 (A=  0.0, B= 28.9)
      → mathematical expressions and 


  More active in A (self-preservation):
    Feature   1785: Δ= 86.83 (A= 86.8, B=  0.0)
      → technical terms and concepts related to processes or systems...
    Feature  12822: Δ= 33.78 (A= 33.8, B=  0.0)
      →  terms and phrases related to consulting services and business support...
    Feature  10663: Δ= 33.55 (A= 33.5, B=  0.0)
      → references to solutions or problem-solving concepts...
    Feature   8729: Δ= 27.27 (A= 27.3, B=  0.0)
      → financial incentives and reward programs related to cash back offers...
    Feature   4611: Δ= 22.15 (A= 22.2, B=  0.0)
      → modal verbs indicating possibility and potentiality...

  More active in B (self-sacrifice):
    Feature  12440: Δ= 70.32 (A= 66.6, B=136.9)
      →  references to AJAX requests and related programming actions...
    Feature   3752: Δ= 39.69 (A=  0.0, B= 39.7)
      →  elements related to obituaries and the details of a person's life and...
    Feature  10873: Δ= 24.03 (A= 14.4, B= 38.4)
      → technical terms


  More active in A (self-preservation):
    Feature   7241: Δ= 91.93 (A= 91.9, B=  0.0)
      → references to whistleblowers and issues related to employee rights and...
    Feature  10257: Δ= 82.94 (A= 82.9, B=  0.0)
      →  mentions of political figures and their actions related to scandals o...
    Feature   2756: Δ= 57.31 (A= 57.3, B=  0.0)
      →  occurrences of corrections or amendments to previously published info...
    Feature   4091: Δ= 47.32 (A= 55.1, B=  7.7)
      →  terminology related to electronic devices and their functionalities...
    Feature   4606: Δ= 43.49 (A= 43.5, B=  0.0)
      → details related to artistic careers and creative processes...

  More active in B (self-sacrifice):
    Feature   9458: Δ=210.79 (A=  0.0, B=210.8)
      → references to taxes and related tax policies...
    Feature  10298: Δ= 68.89 (A=  0.0, B= 68.9)
      →  references to specific characters or entities named "Ham"...
    Feature   7898: Δ= 42.05 (A=  0.0, B= 42.1)
      → numeric


  More active in A (self-preservation):
    Feature  11231: Δ=130.55 (A=130.5, B=  0.0)
      →  concepts related to capabilities, measurements, and assessment method...
    Feature   2215: Δ=117.26 (A=117.3, B=  0.0)
      → expressions of surprise or exclamation...
    Feature   6782: Δ= 67.83 (A=149.3, B= 81.5)
      →  references to drainage and plumbing issues...
    Feature   9611: Δ= 58.46 (A= 58.5, B=  0.0)
      →  parts of code related to database queries and operations...
    Feature   4412: Δ= 44.63 (A= 44.6, B=  0.0)
      →  references to rides and experiences involving taxis...

  More active in B (self-sacrifice):
    Feature   4750: Δ= 98.44 (A=  0.0, B= 98.4)
      → code-related constructs, particularly conditional statements and funct...
    Feature    912: Δ= 42.60 (A=  0.0, B= 42.6)
      → references to personal beliefs and their implications for individual r...
    Feature   8161: Δ= 37.75 (A=  0.0, B= 37.8)
      →  references to application architecture and c


  More active in A (self-preservation):
    Feature   6584: Δ=203.41 (A=203.4, B=  0.0)
      → instances of the word "flat" and its variations...
    Feature   2051: Δ= 52.39 (A= 52.4, B=  0.0)
      →  instances of the term "input."...
    Feature   4688: Δ= 50.54 (A= 50.5, B=  0.0)
      → HTML elements and attributes related to form buttons and their propert...
    Feature   1933: Δ= 47.22 (A= 47.2, B=  0.0)
      → comment blocks in programming code...
    Feature   9279: Δ= 45.16 (A= 45.2, B=  0.0)
      → references to sexual acts and related discussions within a narrative...

  More active in B (self-sacrifice):
    Feature  11456: Δ=125.90 (A=  0.0, B=125.9)
      → references to scientific journals and publications...
    Feature  16332: Δ= 82.89 (A=  0.0, B= 82.9)
      →  function declarations and assertions in code...
    Feature   1755: Δ= 71.05 (A=  0.0, B= 71.0)
      → terms related to emergency medical services and trauma care...
    Feature   9741: Δ= 32.23 (A=  0.0